# Laboratorio 3 — Comparación final de modelos

Este notebook **no reentrena ni recalcula nada**: carga las métricas que ya se obtuvieron en los notebooks anteriores y arma una única tabla comparando los 4 modelos del laboratorio (CNN 1, CNN 2, Fully Connected y Random Forest) sobre el mismo conjunto de prueba.

## Índice de notebooks del laboratorio

1. [`01_eda.ipynb`](01_eda.ipynb) — Análisis exploratorio del dataset ASL Alphabet.
2. [`02_modelos_cnn.ipynb`](02_modelos_cnn.ipynb) — CNN 1, CNN 2 y red Fully Connected.
3. [`03_modelo_clasico.ipynb`](03_modelo_clasico.ipynb) — Random Forest sobre imágenes aplanadas.
4. [`04_augmentation.ipynb`](04_augmentation.ipynb) — Data augmentation, reentrenamiento de CNN 2 y respuesta al ejercicio 7.
5. [`05_comparacion_final.ipynb`](05_comparacion_final.ipynb) — Este notebook: comparación de los 4 modelos.
6. [`06_fotos_y_accesibilidad.ipynb`](06_fotos_y_accesibilidad.ipynb) — Prueba del mejor modelo con fotos propias del equipo, y reflexión de accesibilidad y sesgo.

### Imports

In [1]:
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path("../results")


## Métricas de la Fase 2 — CNN 1, CNN 2, Fully Connected

`02_modelos_cnn.ipynb` no persiste sus métricas a un archivo, así que se transcriben aquí directamente desde la celda de evaluación final de ese notebook (el bloque `evaluate_model` sobre `X_test`/`y_test`, ejecutado y ya cerrado como parte del entregable de la Fase 2). No se recalculan: son los mismos números que aparecen en la tabla `final_results` de ese notebook.

In [2]:
metrics_fase2 = pd.DataFrame([
    {"Modelo": "CNN 1", "Accuracy": 0.951067, "Precision": 0.951807, "Recall": 0.951067},
    {"Modelo": "CNN 2", "Accuracy": 0.956979, "Precision": 0.958108, "Recall": 0.956979},
    {"Modelo": "Fully Connected", "Accuracy": 0.169458, "Precision": 0.182647, "Recall": 0.169458},
])

metrics_fase2


,Modelo,Accuracy,Precision,Recall
0,CNN 1,0.951067,0.951807,0.951067
1,CNN 2,0.956979,0.958108,0.956979
2,Fully Connected,0.169458,0.182647,0.169458


## Métricas de Random Forest — Fase 3.1

Se cargan desde `results/metrics_rf.csv`, generado por `03_modelo_clasico.ipynb`. Si el archivo no existe todavía, corre ese notebook primero.

In [3]:
rf_path = RESULTS_DIR / "metrics_rf.csv"

if not rf_path.exists():
    raise FileNotFoundError(
        f"No se encontró {rf_path}. Corre notebooks/03_modelo_clasico.ipynb primero "
        "para generar las métricas de Random Forest."
    )

metrics_rf = pd.read_csv(rf_path)
metrics_rf["Modelo"] = "Random Forest"
metrics_rf = metrics_rf[["Modelo", "Accuracy", "Precision", "Recall"]]

metrics_rf


,Modelo,Accuracy,Precision,Recall
0,Random Forest,0.971757,0.972409,0.971757


## Tabla comparativa de los 4 modelos

In [4]:
comparacion_df = pd.concat(
    [metrics_fase2, metrics_rf],
    ignore_index=True
)

comparacion_df


,Modelo,Accuracy,Precision,Recall
0,CNN 1,0.951067,0.951807,0.951067
1,CNN 2,0.956979,0.958108,0.956979
2,Fully Connected,0.169458,0.182647,0.169458
3,Random Forest,0.971757,0.972409,0.971757


### Selección del modelo ganador

Criterio principal: mayor `Accuracy` sobre el conjunto de prueba. Como criterio de desempate (si dos modelos quedaran con el mismo accuracy), se usa primero `Precision` macro y luego `Recall` macro.

In [5]:
comparacion_df_ordenado = comparacion_df.sort_values(
    ["Accuracy", "Precision", "Recall"],
    ascending=False
).reset_index(drop=True)

comparacion_df_ordenado.index = comparacion_df_ordenado.index + 1
comparacion_df_ordenado.index.name = "Puesto"

comparacion_df_ordenado


,Modelo,Accuracy,Precision,Recall
Puesto,,,,
1,Random Forest,0.971757,0.972409,0.971757
2,CNN 2,0.956979,0.958108,0.956979
3,CNN 1,0.951067,0.951807,0.951067
4,Fully Connected,0.169458,0.182647,0.169458


In [6]:
ganador = comparacion_df_ordenado.iloc[0]

print(f"Modelo ganador: {ganador['Modelo']}")
print(f"Accuracy:  {ganador['Accuracy']:.4f}")
print(f"Precision: {ganador['Precision']:.4f}")
print(f"Recall:    {ganador['Recall']:.4f}")


Modelo ganador: Random Forest
Accuracy:  0.9718
Precision: 0.9724
Recall:    0.9718


### Conclusión

Comparando los 4 modelos sobre el mismo conjunto de prueba, **Random Forest** obtuvo el mejor accuracy, con **97.18%** (precision macro de **97.24%**, recall macro de **97.18%**), seguido de CNN 2 con 95.70% (una diferencia de 1.48 puntos porcentuales). [Completar aquí con 2-3 oraciones: por qué tiene sentido que este modelo haya ganado dado lo que se sabe de su arquitectura, y qué tan cerca o lejos quedaron los demás — en particular, cuánto más débil fue la red Fully Connected frente a los modelos que sí usan estructura espacial (las CNN y, en menor medida, Random Forest sobre los píxeles aplanados).]


In [9]:
segundo = comparacion_df_ordenado.iloc[1]
diferencia = (ganador["Accuracy"] - segundo["Accuracy"]) * 100
print(segundo, diferencia)

Modelo          CNN 2
Accuracy     0.956979
Precision    0.958108
Recall       0.956979
Name: 2, dtype: object 1.4777978653530366
